# Comprehensive Synapse Backtest (DJIA 2015-2024)
**Rolling Walk-Forward Validation**

This notebook implements a robust backtesting framework for the Synapse Arbitrator system.
It uses a rolling window approach to validate performance on the Dow Jones Industrial Average (DJIA) constituents over a 10-year period.

In [1]:
# Cell 1: Setup & Dependencies
!pip install -q --upgrade yfinance stockstats
!pip install -q torch pandas numpy scipy matplotlib

import os
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from scipy.special import softmax
from stockstats import StockDataFrame
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✓ Setup complete. Device: {device}")

✓ Setup complete. Device: cuda


In [2]:
# Cell 2: Configuration
class Config:
    # Data Universe (DJIA 30)
    TICKERS = [
        "AAPL", "MSFT", "JPM", "V", "RTX", "PG", "GS", "NKE", "DIS", "AXP",
        "HD", "INTC", "WMT", "IBM", "MRK", "UNH", "KO", "CAT", "TRV", "JNJ",
        "CVX", "MCD", "VZ", "CSCO", "XOM", "BA", "MMM", "PFE", "WBA", "DD"
    ]
    
    # Date Ranges
    START_DATE = "2015-01-01"
    END_DATE = "2024-12-31"
    
    # Rolling Window Settings
    TRAIN_WINDOW_SIZE = 365 * 2  # 2 Years
    TEST_WINDOW_SIZE = 365 // 2  # 6 Months
    ROLLING_STEP_SIZE = 365 // 2 # 6 Months
    
    # Environment Settings
    INITIAL_CAPITAL = 1_000_000
    
    # Model Hyperparameters
    TRAIN_EPOCHS = 30
    LEARNING_RATE = 1e-3
    GAMMA = 0.99
    
    # Technical Indicators
    INDICATORS = ['macd', 'rsi_14', 'cci_14', 'dx_14']

config = Config()
print(f"Configured for {len(config.TICKERS)} tickers over {config.TRAIN_WINDOW_SIZE//365}y train / {config.TEST_WINDOW_SIZE//30}m test windows.")

Configured for 30 tickers over 2y train / 6m test windows.


In [3]:
# Cell 3: Data Loader
def download_data():
    """Downloads data for tickers specified in config."""
    print(f"Downloading data for {len(config.TICKERS)} tickers from {config.START_DATE} to {config.END_DATE}...")
    
    df_list = []
    for tic in config.TICKERS:
        try:
            data = yf.download(tic, start=config.START_DATE, end=config.END_DATE, progress=False, auto_adjust=False)
            if data.empty:
                print(f"  Warning: No data for {tic}")
                continue
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = [col[0] for col in data.columns]
            data = data.reset_index()
            data['tic'] = tic
            data = data.rename(columns={'Date': 'date', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close': 'close', 'Adj Close': 'adj_close', 'Volume': 'volume'})
            if 'adj_close' not in data.columns:
                data['adj_close'] = data['close']
            data = data[['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'tic']]
            df_list.append(data)
        except Exception as e:
            print(f"  ✗ Error downloading {tic}: {e}")

    if not df_list:
        raise ValueError("No data downloaded!")

    df = pd.concat(df_list, ignore_index=True)
    df['date'] = pd.to_datetime(df['date'])
    return df

def preprocess_data(df):
    """Adds technical indicators."""
    print("Adding technical indicators...")
    result = []
    for tic in df['tic'].unique():
        tic_df = df[df['tic'] == tic].copy().sort_values('date').reset_index(drop=True)
        stock = StockDataFrame.retype(tic_df[['date', 'open', 'high', 'low', 'close', 'volume']].copy())
        for ind in config.INDICATORS:
            tic_df[ind] = stock[ind].values
        result.append(tic_df)

    df_processed = pd.concat(result, ignore_index=True)
    for ind in config.INDICATORS:
        df_processed[ind] = df_processed.groupby('tic')[ind].transform(lambda x: x.ffill().bfill())
    df_processed = df_processed.dropna()
    print(f"✅ Final dataset: {len(df_processed)} rows")
    return df_processed

def get_rolling_windows(df):
    """Generates train/test splits."""
    dates = sorted(df['date'].unique())
    total_days = len(dates)
    windows = []
    current_idx = 0
    
    while current_idx + config.TRAIN_WINDOW_SIZE + config.TEST_WINDOW_SIZE <= total_days:
        train_start_idx = current_idx
        train_end_idx = current_idx + config.TRAIN_WINDOW_SIZE
        test_start_idx = train_end_idx
        test_end_idx = test_start_idx + config.TEST_WINDOW_SIZE
        
        windows.append({
            'train_start': dates[train_start_idx],
            'train_end': dates[train_end_idx - 1],
            'test_start': dates[test_start_idx],
            'test_end': dates[test_end_idx - 1]
        })
        current_idx += config.ROLLING_STEP_SIZE
    return windows

# Execute Data Load
df = download_data()
df = preprocess_data(df)
windows = get_rolling_windows(df)
print(f"Generated {len(windows)} rolling windows.")

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WBA"}}}
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['WBA']: YFTzMissingError('possibly delisted; no timezone found')


Adding technical indicators...
✅ Final dataset: 72935 rows
Generated 9 rolling windows.


In [4]:
# Cell 4: Models (Env, Agent, Synapse)
class TradingEnv:
    def __init__(self, df, initial_capital=config.INITIAL_CAPITAL):
        self.df = df.sort_values(['date', 'tic']).reset_index(drop=True)
        self.dates = sorted(df['date'].unique())
        self.tickers = sorted(df['tic'].unique())
        self.n_stocks = len(self.tickers)
        self.initial_capital = initial_capital
        self.n_features_per_stock = 1 + len(config.INDICATORS) 
        self.state_dim = 1 + self.n_stocks * self.n_features_per_stock
        self.action_dim = self.n_stocks
        
    def reset(self):
        self.day_idx = 0
        self.cash = self.initial_capital
        self.holdings = np.zeros(self.n_stocks)
        self.portfolio_values = [self.initial_capital]
        return self._get_state()
    
    def _get_state(self):
        day = self.dates[self.day_idx]
        day_data = self.df[self.df['date'] == day]
        features = [self.cash / self.initial_capital]
        prices = []
        for tic in self.tickers:
            row = day_data[day_data['tic'] == tic]
            if len(row) > 0:
                price = row['close'].values[0]
                prices.append(price)
                feat = [price/100.0] 
                for ind in config.INDICATORS:
                    val = row[ind].values[0]
                    if ind in ['rsi_14', 'dx_14', 'cci_14']: val /= 100.0
                    feat.append(val)
                features.extend(feat)
            else:
                prices.append(0)
                features.extend([0] * self.n_features_per_stock)
        self.current_prices = np.array(prices)
        return np.array(features, dtype=np.float32)
    
    def step(self, action):
        action = np.clip(action, -1, 1)
        pv_before = self.cash + np.sum(self.holdings * self.current_prices)
        target_holdings = (action * 0.5 + 0.5) * pv_before / (self.current_prices + 1e-8)
        self.holdings = target_holdings
        self.cash = pv_before - np.sum(self.holdings * self.current_prices)
        self.day_idx += 1
        done = self.day_idx >= len(self.dates) - 1
        state = self._get_state() if not done else np.zeros(self.state_dim)
        pv_after = self.cash + np.sum(self.holdings * self.current_prices)
        reward = (pv_after - pv_before) / pv_before
        self.portfolio_values.append(pv_after)
        return state, reward, done, {}

class SimpleAgent:
    def __init__(self, state_dim, action_dim, seed=42):
        torch.manual_seed(seed)
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, action_dim), nn.Tanh()
        ).to(device)
        self.opt = torch.optim.Adam(self.net.parameters(), lr=config.LEARNING_RATE)
        self.action_dim = action_dim
        
    def act(self, state):
        with torch.no_grad():
            return self.net(torch.FloatTensor(state).to(device)).cpu().numpy()
    
    def train_episode(self, env):
        states, actions, rewards = [], [], []
        state, done = env.reset(), False
        while not done:
            action = self.act(state) + np.random.normal(0, 0.1, self.action_dim)
            action = np.clip(action, -1, 1)
            next_state, reward, done, _ = env.step(action)
            states.append(state); actions.append(action); rewards.append(reward)
            state = next_state
        returns = []
        R = 0
        for r in reversed(rewards):
            R = r + config.GAMMA * R
            returns.insert(0, R)
        returns = torch.FloatTensor(returns).to(device)
        if len(returns) > 1: returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        pred = self.net(torch.FloatTensor(np.array(states)).to(device))
        loss = -((pred * torch.FloatTensor(np.array(actions)).to(device)).sum(1) * returns).mean()
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        return sum(rewards)

class SynapseArbitrator:
    def __init__(self, agents, window=20):
        self.agents = agents
        self.profits = [[] for _ in agents]
        self.window = window
    def predict(self, state):
        scores = [np.mean(p[-self.window:]) if p else 0 for p in self.profits]
        weights = softmax(np.array(scores))
        actions = [a.act(state) for a in self.agents]
        return sum(w * a for w, a in zip(weights, actions)), weights
    def update(self, reward):
        for p in self.profits: p.append(reward)

print("✓ Models defined")

✓ Models defined


In [ ]:
# Cell 5: Backtest Engine
def evaluate(agent, env):
    state, done, rewards = env.reset(), False, []
    while not done:
        if hasattr(agent, 'act'): action = agent.act(state)
        else: action, _ = agent.predict(state)
        state, reward, done, _ = env.step(action)
        if hasattr(agent, 'update'): agent.update(reward)
        rewards.append(reward)
    if np.std(rewards) < 1e-8: sharpe = 0
    else: sharpe = np.mean(rewards) / np.std(rewards) * np.sqrt(252)
    total_return = (env.portfolio_values[-1] / env.portfolio_values[0] - 1) * 100
    return {'sharpe': sharpe, 'return': total_return, 'final_value': env.portfolio_values[-1]}

results = []
print(f"Starting backtest with {len(windows)} rolling windows...")

for i, w in enumerate(windows):
    print(f"\n=== Window {i+1}/{len(windows)} ===")
    print(f"Train: {w['train_start'].date()} -> {w['train_end'].date()}")
    print(f"Test:  {w['test_start'].date()} -> {w['test_end'].date()}")
    
    train_df = df[(df['date'] >= w['train_start']) & (df['date'] <= w['train_end'])].copy()
    test_df = df[(df['date'] >= w['test_start']) & (df['date'] <= w['test_end'])].copy()
    
    if len(train_df) < 100 or len(test_df) < 10:
        print("⚠ Skipping window - insufficient data")
        continue
        
    train_env = TradingEnv(train_df)
    test_env = TradingEnv(test_df)
    
    print("Training agents...")
    agents = []
    for j in range(3):
        agent = SimpleAgent(train_env.state_dim, train_env.action_dim, seed=42+j*100)
        for ep in range(config.TRAIN_EPOCHS):
            agent.train_episode(train_env)
        agents.append(agent)
        
    synapse = SynapseArbitrator(agents)
    
    print("Evaluating...")
    single = evaluate(agents[0], test_env)
    synapse_res = evaluate(synapse, test_env)
    
    print(f"  Single:  Sharpe={single['sharpe']:.3f}, Return={single['return']:.1f}%")
    print(f"  Synapse: Sharpe={synapse_res['sharpe']:.3f}, Return={synapse_res['return']:.1f}%")
    
    results.append({
        'window_id': i+1,
        'test_start': w['test_start'],
        'single_sharpe': single['sharpe'],
        'synapse_sharpe': synapse_res['sharpe'],
        'single_return': single['return'],
        'synapse_return': synapse_res['return']
    })

Starting backtest with 9 rolling windows...

=== Window 1/9 ===
Train: 2015-01-02 -> 2017-11-22
Test:  2017-11-24 -> 2018-08-15
Training agents...


In [ ]:
# Cell 6: Analysis
results_df = pd.DataFrame(results)
if not results_df.empty:
    print("\n=== Summary Metrics ===")
    print(f"Average Single Sharpe:  {results_df['single_sharpe'].mean():.3f}")
    print(f"Average Synapse Sharpe: {results_df['synapse_sharpe'].mean():.3f}")
    print(f"Net Improvement:        {results_df['synapse_sharpe'].mean() - results_df['single_sharpe'].mean():+.3f}")
    
    plt.figure(figsize=(12, 6))
    x = range(len(results_df))
    width = 0.35
    plt.bar([i - width/2 for i in x], results_df['single_sharpe'], width, label='Single Agent', color='gray')
    plt.bar([i + width/2 for i in x], results_df['synapse_sharpe'], width, label='Synapse', color='green')
    plt.xlabel('Window')
    plt.ylabel('Sharpe Ratio')
    plt.title('Synapse vs Single Agent - Rolling Backtest (DJIA 2015-2024)')
    plt.xticks(x, results_df['test_start'].dt.strftime('%Y-%m'), rotation=45)
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
    
    results_df.to_csv('comprehensive_results.csv', index=False)
    print("Results saved to comprehensive_results.csv")